In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

print("python:", sys.executable)

ROOT = Path.cwd().parent
DATA = ROOT / "data" / "raw"
print("project root:", ROOT)
print([p.relative_to(DATA).as_posix() for p in sorted(DATA.glob("*/*.parquet"))])

python: c:\Projects\aml-fraud-detection\.venv\Scripts\python.exe
project root: c:\Projects\aml-fraud-detection
['benchmark/test.parquet', 'benchmark/train.parquet', 'benchmark/valid.parquet', 'chronological/test.parquet', 'chronological/train.parquet', 'chronological/valid.parquet']


In [2]:
%load_ext autoreload
%autoreload 2

sys.path.insert(0, str(ROOT))
from src.data import load_split
from src.features import make_features

valid = load_split("benchmark", "valid")
print("raw:", valid.shape, "| fraud:", int(valid["is_fraud"].sum()))
print("raw RAM (MB):", round(valid.memory_usage(deep=True).sum() / 1e6))

X_valid = make_features(valid)
print("features:", X_valid.shape)
print("feature RAM (MB):", round(X_valid.memory_usage(deep=True).sum() / 1e6))
print(list(X_valid.columns))

raw: (1800359, 15) | fraud: 1311
raw RAM (MB): 70
features: (1800359, 23)
feature RAM (MB): 74
['log_amount', 'log_time_since_last', 'log_velocity', 'spending_deviation_score', 'burst', 'has_history', 'hour_sin', 'hour_cos', 'day_of_week', 'is_first_transaction', 'is_weekend', 'is_cross_bank_transfer', 'is_cross_currency_transfer', 'is_new_receiver', 'is_new_bank', 'is_new_payment_format', 'channel_ACH', 'channel_Credit Card', 'channel_Cheque', 'channel_Bitcoin', 'channel_Reinvestment', 'channel_Wire', 'channel_Cash']


In [3]:
from src.data import load_split
from src.features import make_features, FEATURE_COLUMNS, CHANNELS

train = load_split("benchmark", "train")
X_train = make_features(train)

print("columns in expected order:", list(X_train.columns) == FEATURE_COLUMNS)

chan = X_train[[f"channel_{c}" for c in CHANNELS]]
print("every row has exactly one channel:", bool((chan.sum(axis=1) == 1).all()))
print()
print("rows per channel:")
print(chan.sum().astype(int))
print()
print("fraud total:", int(train["is_fraud"].sum()))
print("fraud in ACH:", int(train.loc[X_train["channel_ACH"] == 1, "is_fraud"].sum()))
print("RAM raw / features (MB):",
      round(train.memory_usage(deep=True).sum() / 1e6),
      "/", round(X_train.memory_usage(deep=True).sum() / 1e6))

columns in expected order: True
every row has exactly one channel: True

rows per channel:
channel_ACH              977407
channel_Bitcoin          318526
channel_Cash             802058
channel_Cheque          3057445
channel_Credit Card     2173776
channel_Reinvestment     791659
channel_Wire             280804
dtype: int64

fraud total: 6119
fraud in ACH: 4954
RAM raw / features (MB): 328 / 344
